# Task 2: read prediction (accuracy ↑)

**Shipped:** pairwise logistic regression on φ(u, item1) − φ(u, item2), with
φ = [BPR, log popularity, cosine sum/max/mean to history] (`TASK2_FEATURES`),
then test-design rules (unseen item: guard, 0% of rows; seen by user: 1.92% of rows).
**Val 0.9371 (held-out half) | public 0.9302 → + seen-by-user rule (expected ~0.940)**.
Tiers: 3.5 ≥ 0.93, 4.0 ≥ 0.94.

**Data facts:** decoys popularity-matched; every test candidate appears in training;
the test's decoy sampler does not exclude the user's history (rules.ipynb).

Related: `task2_diagnostics.ipynb` (error analysis), `rules.ipynb` (rule evidence).

In [ ]:
%load_ext autoreload
%autoreload 2
 
import itertools
 
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
 
from assignment_1.modules.models import bpr_scorer, fit_bpr
from assignment_1.modules.tasks.task2 import TASK2_BPR_PARAMS, TASK2_FEATURES, task2_pair_features
from assignment_1.modules.util import calc_pair_accuracy, load_csv, make_pair_validation, predict_pairs

In [ ]:
DATA_DIR = "../data"  # notebook runs from notebooks/
 
train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_2_test.csv", DATA_DIR)

## Data checks
Test decoys are popularity-matched if item1/item2 medians match the "∝ interactions" reference.

In [ ]:
pop = train["item"].value_counts()
p1, p2 = test["item1"].map(pop).fillna(0), test["item2"].map(pop).fillna(0)
print(f"unseen users {(~test.user.isin(train.user)).mean():.1%} | "
      f"unseen item1 {(p1 == 0).mean():.1%}, item2 {(p2 == 0).mean():.1%}")
print(f"median popularity: item1 {p1.median():.0f}, item2 {p2.median():.0f} | "
      f"refs: uniform {pop.median():.0f}, ∝ interactions {train['item'].map(pop).median():.0f}")
print(f"P(item1 more popular) {(p1 > p2).mean():.3f} | ties {(p1 == p2).mean():.3f}")

,user,item,timestamp,rating
0,u6615992338,i4026940741,1387439652,4
1,u1474879069,i5361500840,1358838765,5
2,u1060791114,i1810200320,1429854010,5
3,u1309139104,i4536388280,1409468967,4
4,u3188006761,i2217243877,1439792233,3


## Validation pairs
Same call as `run_task2`: held-out interactions + popularity-sampled decoys (history excluded).
The combiner is fit on `half` and scored on `~half`.

In [ ]:
tr, pairs = make_pair_validation(train, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
print(f"{len(pairs):,} validation pairs")

## E1. Reference scorers (full validation)
**Result:** random ≈ 0.50, constant 0.50 (ties → item1), popularity ≈ 0.50 (decoys matched),
baseline BPR (5 factors) ≈ 0.877.

In [ ]:
rng = np.random.default_rng(0)
pd.Series({
    "random": calc_pair_accuracy(pairs, lambda u, i: rng.random(len(i))),
    "constant": calc_pair_accuracy(pairs, lambda u, i: np.zeros(len(i))),
    "popularity": calc_pair_accuracy(pairs, lambda u, i: pd.Series(i).map(pop).fillna(0).to_numpy()),
    "BPR 5 factors": calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(
        tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100))),
}).round(4)

,user,item1,item2,which
0,u6589235933,i6679693485,i3513531206,NaN
1,u5036346542,i9742011999,i4000434505,NaN
2,u2543223670,i8553129811,i8159127157,NaN
3,u6680502243,i5391486098,i3623313194,NaN
4,u2041216719,i5735921315,i5729684422,NaN


## E2. BPR grid
**Result:** best factors 100, regularization 0.003 → `TASK2_BPR_PARAMS`.
learning_rate / iterations never swept (always 0.01 / 100).

In [ ]:
def bpr_grid(grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        rows.append({**params, "val_acc": calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(tr, **params)))})
    return pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)

In [ ]:
# bpr_grid({"factors": [50, 100, 200], "regularization": [0.003, 0.01, 0.03],
#           "learning_rate": [0.01], "iterations": [100]})

## E3. Pairwise combiner: which features help? (held-out half)
**Result:** shipped BPR alone 0.9146 | BPR + pop 0.9146 (no gain: decoys popularity-matched)
| + cosine LR 0.9371, GB 0.9373 (trees add nothing → ship LR, the course model).

In [ ]:
X = task2_pair_features(tr, pairs)            # packaged features, columns = TASK2_FEATURES
col = {c: k for k, c in enumerate(TASK2_FEATURES)}

def evaluate(cols: list[str]) -> dict:
    Xc = X[:, [col[c] for c in cols]]
    out = {}
    for name, m in [("lr", LogisticRegression(max_iter=1000)),
                    ("gb", HistGradientBoostingClassifier(random_state=0))]:
        m.fit(Xc[half], y[half])
        out[name] = round((m.predict(Xc[~half]) == y[~half]).mean(), 4)
    return out

b_pred = predict_pairs(pairs, bpr_scorer(fit_bpr(tr, **TASK2_BPR_PARAMS)))
print(f"BPR alone    {(b_pred == pairs['which'].to_numpy())[~half].mean():.4f}")
print("bpr + pop   ", evaluate(["bpr", "log_pop"]))
print("+ cosine    ", evaluate(TASK2_FEATURES))

## E4. LR coefficients (shipped model)
Sign and size of each feature's weight; features are unscaled, so compare signs, not sizes.

In [ ]:
lr = LogisticRegression(max_iter=1000).fit(X[half], y[half])
pd.Series(lr.coef_[0], index=TASK2_FEATURES).round(3)

kept 100.0% of validation rows


In [ ]:
test4, train4 = data[4]
user_ts = set(train4["user"] + "|" + train4["timestamp"].astype(str))
same_sec = (test4["user"] + "|" + test4["timestamp"].astype(str)).isin(user_ts)
print(f"test rows whose (user, t) matches a train row's exact timestamp: {same_sec.mean():.1%}")

## Next
- Secure 4.0 (≥ 0.94 public): top-k cosine (mean of k largest similarities), rating-weighted
  cosine, recency-weighted cosine (train has timestamps).
- Validation built like the test (decoys may come from the user's history), so the combiner
  can learn the seen-by-user case instead of relying on the rule.